# Getting started

Forecast with two baselines, load the renewal model's stored fit at the same origin, then plot and score all three.

Every figure and table below comes from the Python code shown above it, which calls the package's functions directly.
[Setup](../setup/) installs the package and shows how to run this code in a REPL, a notebook or a script.
`task nowcastbdbv:data` fetches the tidy data into `data/`, and `task bdbvhub:fit-store` fetches the stored renewal fits into `output/fit-store`.
This page is a notebook, and `task getting-started` runs it again.

## Load the data

`load_target` reads the weekly confirmed cases by onset week of every hub vintage.
`load_daily` reads the daily onsets of every hub release, which the baselines need.
`load_delays` reads the reporting delays the hub published, which the renewal models use.

In [1]:
from pathlib import Path

import pandas as pd

from bdbvhub.models import get_model
from bdbvhub.store import store
from nowcastbdbv.data.tidy import load_daily, load_delays, load_target
from nowcastbdbv.data.zones import load_zone_data
from nowcasteval import backtest as bt
from nowcasteval.nowcast import relative_crps, release_backtest
from nowcasteval.plots import plot_location_grid, save_light_dark
from nowcasteval.scoring import relative_skill
from nowcastutils.base import NATIONAL

target = load_target("data")
daily = load_daily("data")
delays = load_delays("data")
real = target.loc[target["kind"] == "real"]
sorted(real["vintage"].unique())

['2026-08-18',
 '2026-10-09',
 '2026-10-17',
 '2026-10-24',
 '2026-11-01',
 '2026-11-08',
 '2026-11-15',
 '2026-11-22']

## Build the forecast task

The newest release scores the forecasts.
The origin is the newest release before it at which the fit store holds a renewal fit, or the second newest release if it holds none.
The task holds what a model sees at the origin, including the daily onsets of every release up to it, and the weeks to forecast.
Its reference date is the Saturday after the origin, and the forecast weeks run from six weeks before it to four weeks after it.
`origin_tasks` builds the task of every real origin as the fit store fits it, so the baselines and the stored renewal fit see the same task.
`current_keys` gives the key of each renewal fit of this code and data, from the tasks with the health-zone tables (`load_zone_data`), as the store keys them.
`newest_fit` returns the fit `bdbvhub fit-store load` uses at the newest origin that has one.
That is the fit of the current key, or, until that fit is made, the newest fit of older code.

In [2]:
releases = bt.real_vintages(target)
LATER = releases[-1]

config = bt.load_config("packages/nowcastbdbv/config/backtest.toml")
tasks = store.origin_tasks(target, delays, config, daily)
zones = load_zone_data("data")
zoned = store.origin_tasks(target, delays, config, daily, zones)
keys = store.current_keys(
    {o: t for o, t in zoned.items() if o < LATER},
    ["renewal"],
    store.samplers(),
    store.code_digest("renewal"),
)
stored = store.newest_fit("output/fit-store", keys, "renewal")
ORIGIN = releases[-2] if stored is None else stored.meta["origin"]
task = tasks[ORIGIN]
print(task.vintage, task.reference_date, len(task.locations), "provinces")
print(task.target_end_dates[0], "to", task.target_end_dates[-1])
print("scored on", LATER)

2026-11-15 2026-11-21 26 provinces
2026-10-10 to 2026-12-19
scored on 2026-11-22


## Forecast with the baselines

`get_model` returns a model from the registry by name.
`randomwalk` carries the last complete week forward with random walk noise on the log scale.
`randomwalknowcast` nowcasts the recent weeks first ([Baseline models](../methods/baselines/)).
`randomwalknowcast` is the baseline we submit to the hub as `seabbs-rwnowcast`.
`run_forecasts` runs each model, checks the samples, adds the national total and returns 100 trajectories per location.
`learnt` collects each model's modelled delay, which scoring uses.

In [3]:
models = [get_model("randomwalk"), get_model("randomwalknowcast")]
learnt: list[pd.DataFrame] = []
baselines = bt.run_forecasts(models, [task], learnt=learnt)
baselines.groupby("model")["sample"].nunique()

model
randomwalk           100
randomwalknowcast    100
Name: sample, dtype: int64

## Fit the renewal model

The [renewal model](../methods/model/) is fitted by MCMC (NUTS), which takes about half an hour, so this page does not fit it.
The [fit store](../development/cli/#fit-store) fits it in CI at every real origin and stores the result.
This is how it is fitted, with the sampler settings of the submission:

```python
fitted = store.fit(
    "renewal", task, store.samplers(), store.code_digest("renewal")
)
store.write_fit("output/fit-store", fitted)
```

The page loads the stored fit instead.

## Load the stored renewal fit

`newest_fit` read the fit chosen above into tables.
Its `forecasts` table has the same columns as the baselines' forecasts, and its `delays` table holds the fit's modelled delay.
If the store holds no renewal fit, the page says so and shows the baselines only.

In [4]:
if stored is None:
    print("The fit store holds no renewal fit, so only the baselines follow.")
    forecasts, fit_delays = baselines, []
else:
    current = stored.meta["key"] == keys[("renewal", ORIGIN)]
    print("fitted at commit", stored.meta["commit"][:7])
    print("fit of this code" if current else "fit of older code")
    forecasts = pd.concat([baselines, stored.forecasts], ignore_index=True)
    fit_delays = [stored.delays]
forecasts.groupby("model")["sample"].nunique()

fitted at commit 21a6db4
fit of older code


model
randomwalk           100
randomwalknowcast    100
renewal              100
Name: sample, dtype: int64

## Plot the nowcast

Recent weeks are still being reported, so the weeks before the reference date are nowcasts.
The first figure keeps only those weeks, to show what the nowcast adds to the data at the origin.
Each panel shows the data at the origin and as later reported, with the forecast fans ([reading the figures](../methods/evaluation/#reading-the-figures)).
The panels are the national total and the three provinces with the most cases.
`save_light_dark` writes a light and a dark version of each figure.

In [5]:
LATEST = real.loc[real["vintage"] == LATER]
provinces = (
    LATEST.loc[LATEST["location"] != NATIONAL]
    .groupby("location")["total"]
    .sum()
    .nlargest(3)
    .index
)
panels = [NATIONAL, *provinces]
OUT = Path("docs/assets/getting-started")


def save(stem, **kwargs):
    save_light_dark(
        lambda mode: plot_location_grid(
            forecasts,
            target,
            panels,
            truth_vintage=LATER,
            weeks_before=12,
            mode=mode,
            **kwargs,
        ),
        OUT / f"{stem}.svg",
    )


save("nowcast", max_horizon=-1)
save("forecast")
panels

['CD', 'CD-IT', 'CD-NK', 'CD-HU']

![Nowcasts from randomwalk, randomwalknowcast and renewal at the origin against the data as later reported.](../assets/getting-started/nowcast.svg#only-light)
![Nowcasts from randomwalk, randomwalknowcast and renewal at the origin against the data as later reported.](../assets/getting-started/nowcast-dark.svg#only-dark)

## Plot the forecast

The second figure adds the reference week and the three forecast weeks.

![Nowcasts and forecasts from randomwalk, randomwalknowcast and renewal at the origin against the data as later reported.](../assets/getting-started/forecast.svg#only-light)
![Nowcasts and forecasts from randomwalk, randomwalknowcast and renewal at the origin against the data as later reported.](../assets/getting-started/forecast-dark.svg#only-dark)

## Score

Each forecast is scored on what each later release reported ([how scoring works](../methods/evaluation/#real-releases)).
`randomwalk` is scored on its samples as they are, with no delay.
`release_backtest` maps each other model's forecast of final counts to the counts a later release would report, using the model's own modelled delay.
It scores the samples with the continuous ranked probability score (CRPS), on the counts and on the log scale, and interval coverage.
The table gives each model's mean CRPS relative to `randomwalknowcast`, the site's baseline, for the national total and over the provinces.

In [6]:
result = release_backtest(
    [task],
    daily,
    target,
    models=(),
    stored=forecasts,
    delays=pd.concat([*learnt, *fit_delays], ignore_index=True),
)
scores = result.scores.loc[result.scores["target"] == "reported"]
scores = scores.assign(
    level=scores["location"].map(
        lambda x: "National" if x == NATIONAL else "Province"
    )
)
relative_crps(scores, by=["level"], baseline="randomwalknowcast").round(2)

,level,model,n,crps,base_crps,rel_crps,rel_log_crps,coverage_50,coverage_90,bias
0,National,randomwalk,7,60.66,16.86,3.60,3.90,0.43,0.71,-0.04
1,National,randomwalknowcast,7,16.86,16.86,1.00,1.00,0.43,1.00,-0.17
2,National,renewal,7,13.53,16.86,0.80,0.76,0.57,1.00,-0.31
3,Province,randomwalk,182,2.67,0.99,2.69,2.13,0.90,0.92,-0.01
4,Province,randomwalknowcast,182,0.99,0.99,1.00,1.00,0.93,0.98,0.01
5,Province,renewal,182,0.87,0.99,0.87,0.77,0.94,0.98,0.01


The [energy score](../methods/evaluation/#energy-score) scores each scale's forecast of every place and week at once, from the same samples.
The table gives it and its log version relative to `randomwalknowcast`, with $n$ the releases scored.

In [7]:
energy = result.energy.loc[result.energy["target"] == "reported"]
relative_skill(
    energy,
    baseline="randomwalknowcast",
    by=("scale",),
    metrics=("energy", "log_energy"),
    match_on=("origin", "truth_vintage"),
).round(2)

,model,scale,n,relative_energy,relative_log_energy
0,randomwalk,national,1,4.32,5.12
1,randomwalk,provinces,1,3.08,2.04
2,randomwalknowcast,national,1,1.00,1.00
3,randomwalknowcast,provinces,1,1.00,1.00
4,renewal,national,1,0.76,0.73
5,renewal,provinces,1,0.87,0.86


## Next steps

The [API reference](../reference/) documents every function used here, grouped by task.
The [CLI page](../development/cli/) lists the commands the submission workflow runs.
They fetch the data, run the backtest, forecast from the latest vintage and write the hub file.